### Simple Feedback System

The goal of this notebook is to gently introduce a variant of the [CAL theorem](https://arxiv.org/abs/2109.07771) by Lee et al, which is more fine-grained than the original and can be more easily applied to the [Lingua Franca](https://github.com/lf-lang/lingua-franca) coordination language.

Consider the following feedback system.

<img src="../../img/StockExchange.svg" alt="drawing" width="600"/>

#### Max-Plus Operators

In [ ]:
# Max-plus operators are factored out into a shared library, similar to notebooks/ts/max-plus.ts.
import numpy as np

from max_plus import EPS, INF, odot, oplus, otimes, plus, pow_otimes, reset, star, trace
from timeline import write_timeline_to_file

#### Eigenvalue Functions

The max-plus eigenvalue λ(A) can be computed in two equivalent ways:

1. **Trace-based formula**: λ(A) = max{trace(A^k)/k : k = 1, ..., n}

2. **Maximum cycle mean**: λ(A) = max{μ(c) : c is a cycle in G(A)}

where G(A) is the weighted directed graph represented by A, and μ(c) is the mean weight of cycle c.

In [ ]:
# Method 1: Trace-based eigenvalue computation
# λ(A) = max{trace(A^k)/k : k = 1, ..., n}
def eigenvalue_trace(A):
    """Compute max-plus eigenvalue using the trace formula."""
    if not type(A) is np.ndarray or A.ndim != 2:
        raise TypeError("the function only receives 2d numpy array")
    
    eps = float('-inf')
    result = eps  # Start with -inf, not 0!
    
    for k in range(1, len(A) + 1):
        trace_Ak = trace(pow_otimes(A, k))
        if trace_Ak != eps:
            result = max(result, trace_Ak / k)
    
    return result

# Method 2: Graphical eigenvalue computation (maximum cycle mean)
# λ(A) = max{μ(c) : c is a cycle in G(A)}
def find_all_cycles(A):
    """Find all simple cycles in the graph represented by matrix A."""
    n = len(A)
    eps = float('-inf')
    all_cycles = []
    
    # First, find all self-loops (cycles of length 1)
    # These are missed by DFS since path is empty at the start
    for i in range(n):
        if A[i, i] != eps:
            all_cycles.append([i])
    
    # Then find longer cycles using DFS
    def dfs(start, current, path, visited):
        """DFS to find cycles of length > 1 starting from 'start' node."""
        for next_node in range(n):
            # Check if edge exists (from current to next_node)
            # In max-plus matrix A, edge j->i has weight A[i,j]
            if A[next_node, current] == eps:
                continue
            if next_node == start and len(path) > 0:
                # Found a cycle back to start (length > 1)
                all_cycles.append(path + [current])
            elif next_node not in visited:
                visited.add(next_node)
                dfs(start, next_node, path + [current], visited)
                visited.remove(next_node)
    
    # Start DFS from each node to find cycles of length > 1
    for start in range(n):
        dfs(start, start, [], {start})
    
    return all_cycles

def cycle_mean(A, cycle):
    """Compute the mean weight of a cycle."""
    total_weight = 0
    for i in range(len(cycle)):
        from_node = cycle[i]
        to_node = cycle[(i + 1) % len(cycle)]
        # Edge weight from from_node to to_node is A[to_node, from_node]
        total_weight += A[to_node, from_node]
    return total_weight / len(cycle)

def eigenvalue_graphical(A):
    """Compute max-plus eigenvalue using maximum cycle mean."""
    if not type(A) is np.ndarray or A.ndim != 2:
        raise TypeError("the function only receives 2d numpy array")
    
    eps = float('-inf')
    cycles = find_all_cycles(A)
    
    if not cycles:
        return eps  # No cycles means eigenvalue is -∞
    
    max_mean = eps
    for cycle in cycles:
        mean = cycle_mean(A, cycle)
        max_mean = max(max_mean, mean)
    
    return max_mean

# Function for finding eigenvectors of A
def eigenvectors(A):
    """Compute max-plus eigenvectors."""
    if not type(A) is np.ndarray or A.ndim != 2:
        raise TypeError("the function only receives 2d numpy array")
    
    lam = eigenvalue_trace(A)
    # B = A - λ (element-wise subtraction, which is scalar addition in max-plus)
    # This shifts the matrix so eigenvalue becomes 0
    B = A - lam
    
    result = {}
    B_plus = plus(B)
    for i, x in enumerate(B_plus.diagonal()):
        # Check if diagonal element is 0 (with small tolerance for numerical stability)
        if abs(x) < 1e-10:
            result[i] = star(B)[:, i]
    return result

In [ ]:
# Test both eigenvalue methods on a sample matrix
# Note: This test uses a standalone matrix to verify before M is defined

eps = float('-inf')
test_M = np.array([
    [1, eps], 
    [2, 1],
])

print("Test Matrix:")
print(test_M)
print()

# Compute eigenvalue using both methods
lam_trace = eigenvalue_trace(test_M)
lam_graph = eigenvalue_graphical(test_M)

print(f"Eigenvalue (trace method):     {lam_trace}")
print(f"Eigenvalue (graphical method): {lam_graph}")
print(f"Methods agree: {abs(lam_trace - lam_graph) < 1e-10}")
print()

# Show the cycles found
cycles = find_all_cycles(test_M)
print("Cycles found in graph:")
for cycle in cycles:
    mean = cycle_mean(test_M, cycle)
    print(f"  Cycle {cycle}: mean = {mean}")
print()

# Compute eigenvectors
evecs = eigenvectors(test_M)
print("Eigenvectors:")
for idx, vec in evecs.items():
    print(f"  Node {idx}: {vec}")

#### Constants and variables

In [ ]:
# Negative infinity to denote nothing has happened yet.
# Keep lowercase aliases so the rest of the notebook reads like the original derivation.
eps = EPS
inf = INF

# Create vectors. These are row vectors, i.e., shape = 1 x n.
x_p = np.array([[eps] * 2])   # Physical firing times in the k+1-th iteration
x   = np.array([[eps] * 2])   # Physical firing times in the k-th iteration
u   = np.array([[eps] * 1])   # Logical scheduling times of physical actions (which is also physical time)

# Indices
idx_c = 0 # Reaction in reactor Client
idx_e = 1 # Reaction in reactor Exchange

# Setting execution times
# e = np.array([[1, 1]])   # Execution times
e = np.array([[100+100, 200+100]])   # Execution times (reactions + runtime overhead)

#### M and N matrix

In [ ]:
# Evolution matrix
M = np.array([
    [e[0][idx_c], eps], 
    [e[0][idx_c]+e[0][idx_c], e[0][idx_e]],
])

# Control coefficient matrix
N = np.array([
    [0],
    [e[0][idx_c]],
])

In [ ]:
# Eigenvalue
lamb = eigenvalue_graphical(M)
lamb

#### Helper functions

In [ ]:
def step(x, u):
    assert x.shape[0] == 1, "x is not a row vector: x.shape={0}".format(x.shape)
    assert u.shape[0] == 1, "u is not a row vector: u.shape={0}".format(u.shape)
    Mx = otimes(M, x.T)
    Nu = otimes(N, u.T)
    # print("Mx:{}".format(Mx))
    # print("Nu:{}".format(Nu))
    # Returns a row vector to make iterations easy.
    return oplus(Mx, Nu).T

In [ ]:
def report_stats(k, x, u):
    next_logical_time = np.max(u)
    lag_k = x - next_logical_time # lag
    print("Tag index k={k}".format(k=k))
    print("u(k)= {0}".format(u))
    # print("next logical time = {0}".format(next_logical_time))
    print("x(k) = earliest possible firing times = {0}".format(x))
    print("lag(k) = {0}".format(lag_k))

#### A function for modeling user orders

In [ ]:
# Initial firing of actions
action_init_offset = 0
u[0][idx_c] = action_init_offset

# Set the period of the actions 
# (Either same or one is -inf. See NOTE above)
# NOTE 2: eigenvalue(M) = smallest minimum spacing of the physical action to make the systme feasible
action_period = 20 # If actions scheduled simultaneously, use 0.999 to see divergence. Use inf if no future events are scheduled.

# Interation count
k = 0

# Reset x to eps.
x = reset(x)

#### Evolve the system

In [ ]:
# Step the system from k=0 to k=1
x = step(x, u)
k += 1

# Statistics
report_stats(k, x, u)

In [ ]:
# Bump the next physical action firing times
u[0][idx_c] = u[0][idx_c] + action_period

# Step the system from k to k+1
x = step(x, u)
k += 1

# Statistics
report_stats(k, x, u)

## HTML Timeline Export

Generate a Vis.js timeline like the TypeScript Delay notebook. The timeline shows each reaction's earliest firing time `x`, the logical scheduling time `u`, lag `x - u`, and the execution time used by that reaction.

In [ ]:
# Run a fresh multi-step simulation for timeline export.
# This avoids depending on how many times the manual stepping cells above were run.
timeline_x = reset(x)
timeline_u = np.array([[action_init_offset]])

x_history = []
u_history = []
e_history = []

for tag_index in range(10):
    if tag_index > 0:
        timeline_u[0][idx_c] = timeline_u[0][idx_c] + action_period

    timeline_x = step(timeline_x, timeline_u)
    x_history.append(timeline_x.copy())
    u_history.append(float(np.max(timeline_u)))
    e_history.append(e[0].copy())

write_timeline_to_file(
    x_history,
    u_history,
    e_history,
    reaction_names=["Client", "Exchange"],
    filename="../html/stock-exchange-timeline.html",
    title="Stock Exchange Timeline",
    subtitle="Earliest reaction firing times for the Stock Exchange max-plus model.",
)